# 06. LightGBM 개선 실험

판정(05)에서 AI가 규칙 모델을 이겼으니, 이제 AI 쪽을 더 믿을 수 있게 다듬습니다.

| 실험 | 왜 |
| --- | --- |
| ① 단조 제약 | "VIIRS 탐지가 늘면 MODIS 추정도 늘어야 한다"를 강제. 처음 보는 큰 화재에서 엉뚱한 값을 막음 |
| ① Tweedie | 0이 많고 값이 한쪽으로 치우친 탐지 수에 Poisson보다 맞을 수 있음 |
| ② 보정된 오차 범위 | 분위 회귀 구간을 보정해 "80% 구간이 실제로 80%를 맞히게" 함 |
| ③ 모듈별 기여도 | AI가 무엇을 근거로 환산하는지를 규칙 모델의 모듈 단위로 설명 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fsml import evaluate, gbm  # evaluate: 한글 폰트 설정 포함
from fsml.compare import PRIMARY, cross_validate, summarize
from fsml.explain import contributions, feature_importance, module_importance
from fsml.features import make_features, training_rows
from fsml.intervals import ConformalInterval, coverage
from fsml.paths import default_grid
from fsml.rules import CONTINENT_LEVELS, ModularRuleModel
from fsml.splits import auto_folds, auto_split

GRID = default_grid()
grid = pd.read_parquet(GRID)
res = float(np.round(np.diff(np.sort(grid["lat"].unique())).min(), 6))  # 격자 크기(°)
data, FEATURES = make_features(training_rows(grid), res)
kind, folds = auto_folds(data, n=5)
print(f"{GRID.name} | 반복 시험: {'연도별' if kind == 'time' else '지역 블록별'} {len(folds)}회")

## ① LightGBM 변형 비교 (05와 같은 5회 시험)

참고용으로 가장 강했던 규칙 모델도 같이 돌립니다.

In [ ]:
results = cross_validate(
    data, FEATURES, folds,
    models={"규칙 (대륙·위도대·계절)": lambda: ModularRuleModel(levels=CONTINENT_LEVELS)},
    ai_variants=gbm.variants(),
)
summarize(results).round(3)

In [ ]:
wide = results.pivot(index="fold", columns="모델", values=PRIMARY)
ax = wide.plot.bar(figsize=(11, 4), rot=0)
ax.set_ylabel(f"{PRIMARY} (낮을수록 좋음)")
ax.set_title("fold별 합계 오차: LightGBM 변형");

**고르는 법:** 합계 오차가 기본 LightGBM과 비슷하거나 낮으면 **단조 제약 버전을 우선**합니다.
정확도가 같다면 물리적으로 말이 되는 모델이 처음 보는 큰 화재에서 더 안전하기 때문입니다.
바꾸기로 하면 `fsml/gbm.py`에 반영하고 05를 다시 돌립니다.

**결정 (2012–2025 실제 데이터, 연도별 5회):** 단조+Tweedie 153.7 vs Poisson 153.0으로 다섯 해 모두 거의 같아서,
이 규칙대로 **단조+Tweedie를 기본**으로 정했습니다. 그래서 위 표의 `LightGBM`은 이제 단조+Tweedie이고,
나머지는 이전 기본(Poisson)과 단조·Tweedie를 하나씩 뺀 버전입니다. 단조만 건 Poisson은 2024년에 크게 틀려(161.2 ± 62.4) 제외했습니다.

## ② 오차 범위: 보정 전 vs 보정 후 (목표 포함률 80%)

In [ ]:
split_kind, (train, valid, test) = auto_split(data)
interval = ConformalInterval(coverage=0.8).fit(data, FEATURES, train, valid)
X_test, y_test = data.loc[test, FEATURES], data.loc[test, "modis_n"]
pd.DataFrame({
    "분위 회귀 그대로": coverage(y_test, *interval.predict(X_test, calibrated=False)),
    "보정 후 (conformal)": coverage(y_test, *interval.predict(X_test)),
}).T.round(3)

포함률이 80%에 가까울수록 좋습니다. 보정 전이 80%보다 높으면 구간이 쓸데없이 넓은 것이고, 낮으면 과신한 것입니다.
보정 폭(`interval.margin`)이 0에 가까우면 분위 회귀가 이미 잘 맞았다는 뜻입니다.

In [ ]:
print(f"보정 폭: {interval.margin:+.3f} (탐지 수)")

## ③ AI는 무엇을 근거로 환산하나 (모듈별 기여도)

LightGBM의 SHAP 기여도를 규칙 모델과 같은 모듈 단위로 묶었습니다. 발표에서 "AI가 규칙보다 나은 이유"를 설명하는 근거가 됩니다.

In [ ]:
model = gbm.fit(data, FEATURES, train, valid, log_every=0)
# SHAP은 행마다 3,000개 나무를 다 훑어서 느림. 5천 행 표본이면 전체와 비중이 같고(±0.002) 1분 안에 끝남
X_shap = X_test.sample(min(len(X_test), 5_000), random_state=0)
contrib = contributions(model, X_shap)
mods = module_importance(contrib)
ax = mods.sort_values().plot.barh(figsize=(7, 4), color="#ff7a3d")
ax.set_xlabel("평균 |기여도| 비중")
ax.set_title("모듈별 기여도 (시험 세트)")
mods.round(3)

In [ ]:
feature_importance(contrib).round(4).head(12)

## 메모 (TODO)

- 실제 과거 데이터에서 ①을 다시 돌려 기본 설정을 확정
- 토지 피복 열이 붙으면 ③에서 "토지 유형" 모듈 비중 확인, 05에 규칙 (토지유형·위도대·계절) 자동 추가됨
- 달력 합계 단위 구간(지역·월 합계의 오차 범위)은 칸 구간을 더해서는 안 됨 → 지역 단위로 따로 보정 필요